In [ ]:
from sys import path

path.append("..")

import os
from pathlib import Path

os.chdir(Path.cwd().parent)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from src.graph_utils.reading import read_metadata
from src.settings import Settings


In [ ]:
metadata = read_metadata()

In [ ]:

def count_elements(row_array):
    if len(row_array) == 1 and row_array[0] == -1:
        return 0
    return len(row_array)

test_data = pd.read_parquet(Settings.processed_datasets_dir / 'table.parquet')
test_data['count'] = test_data['result'].apply(count_elements)

test_data['percentage'] = test_data.apply(lambda x: x['count'] / metadata[x['dataset_name']]['graph_count'], axis=1)
test_data = test_data.reset_index(drop=True)


test_data['features'] = test_data['features'].astype('string').astype('category')
test_data.columns

In [ ]:
data2d = test_data[
    (~test_data['features'].str.contains('rounded3') 
    & ~(test_data['features'].str.contains(' '))# & ~test_data['features'].str.contains('ldp'))
    & ~test_data['dataset_name'].str.contains('brec_400')
    & ~test_data['dataset_name'].str.contains('regular-2000')
    & ~(test_data['dataset_name'].str.contains(r'graph\d(?!c)') ) # for fair comparison only connected graphs

    # those are unstable
    & ~(
        test_data['features'].str.contains('algebraic_distance(?!_rounded)')
        | test_data['features'].str.contains('katz_index(?!_rounded)')
        | test_data['features'].str.contains('lss')
        | test_data['features'].str.contains('same_community')
        | test_data['features'].str.contains('spanning_edge')
        )
    )

]

data2d = data2d.pivot(index="features", columns="dataset_name", values="percentage")

In [ ]:
list(data2d.index)

In [ ]:
datasets = list(data2d.columns)
datasets

In [ ]:
def power_mean(row, p, epsilon=1e-12):
    row_safe = row + epsilon

    if p == 0:
        return np.exp(np.log(row_safe).mean())
    else:
        return np.mean(row_safe ** p) ** (1 / p)


def metrics(row: np.ndarray, difficulty_weights: np.ndarray) -> np.ndarray:
    mean_score = row.mean()

    p0_score = power_mean(row, p=0)
    p_1_score = power_mean(row, p=-1)
    p_2_score = power_mean(row, p=-2)

    # Size weighted
    sizes = np.array([
        metadata[dataset]['graph_count']
        for dataset in datasets
    ])

    weights_size = sizes / sizes.sum()
    weighted_mean_size = (row * weights_size).sum()

    # Difficulty weighted
    weighted_mean_difficulty = (
        row * difficulty_weights
    ).sum()

    results = np.array([
        mean_score,
        p0_score,
        p_1_score,
        p_2_score,
        weighted_mean_size,
        weighted_mean_difficulty
    ])

    return results

In [ ]:
# A = np.random.randint(0, 101, size=(4,3))
A = 1 - data2d.to_numpy()
A


In [ ]:

# difficulty weighted
dataset_difficulty = 1 - A.mean(axis=0)
difficulty_weights = (
    dataset_difficulty / dataset_difficulty.sum()
)
print(difficulty_weights)

In [ ]:
from itertools import combinations
from tqdm import tqdm
from heapq import heapify, heappop, heappush

In [ ]:
n = A.shape[0]
num_of_metrics = 6
heap_max = 6
best_outputs = [[] for _ in range(num_of_metrics)]

for indecies in tqdm(combinations(range(n), r=3)):

    to_compare = A[list(indecies)]
    outcomes = metrics(np.max(to_compare, axis=0), difficulty_weights).tolist()
    for i in range(num_of_metrics):
        heappush(best_outputs[i], (outcomes[i], indecies))
    # print(indecies, outcomes)

    if len(best_outputs[0]) > heap_max:
        for heap in best_outputs:
            heappop(heap)

best = [max(heap) for heap in best_outputs]
print(*best, sep='\n\n')



In [ ]:
for heap in best_outputs:
    print(*heap, sep='\n')
    print('====')

In [ ]:
from collections import Counter

In [ ]:
trios = [trio for outputs in best_outputs for _, trio in outputs]


descriptors = [desc for outputs in best_outputs for _, trio in outputs for desc in trio]
print(Counter(descriptors))

ranking = Counter(trios)
ranking

In [ ]:
functions = list(data2d.index)

In [ ]:
for trio in ranking:
    print(tuple(map(lambda x: functions[x].replace("['", "").replace("']", ""), trio)), ',')